# Day 19 - Reminder tools (lessons 19-25)

Project from the course: a reminder assistant with three tools.

| Tool | Does |
|---|---|
| `get_current_datetime` | returns now, formatted |
| `add_duration_to_datetime` | adds minutes / hours / days to a datetime |
| `set_reminder` | records a reminder (here: just prints it) |

Flow: send request with `tools` -> Claude replies with `tool_use` blocks -> run the functions -> send `tool_result` blocks back -> repeat until no tool is requested.

In [ ]:
from datetime import datetime, timedelta

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env
client = Anthropic()
MODEL = "claude-sonnet-5-5"

## 1. Tool functions (plain Python, clear errors Claude can read)

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration, unit):
    if unit not in ("minutes", "hours", "days"):
        raise ValueError(f"unit must be minutes, hours or days, got {unit!r}")
    try:
        start = datetime.fromisoformat(datetime_str)
    except ValueError:
        raise ValueError(f"datetime_str must be ISO 8601 (YYYY-MM-DD HH:MM:SS), got {datetime_str!r}")
    return (start + timedelta(**{unit: duration})).strftime("%Y-%m-%d %H:%M:%S")


def set_reminder(content, timestamp):
    if not content:
        raise ValueError("content cannot be empty")
    try:
        datetime.fromisoformat(timestamp)
    except ValueError:
        raise ValueError(f"timestamp must be ISO 8601, got {timestamp!r}")
    print(f"[reminder set] {timestamp} - {content}")
    return f"Reminder set for {timestamp}: {content}"

Quick offline check (no API call):

In [ ]:
now = get_current_datetime()
later = add_duration_to_datetime(now, 2.5, "hours")
print(now, "->", later)
print(set_reminder("team meeting", later))

try:
    add_duration_to_datetime(now, 1, "weeks")
except ValueError as e:
    print("expected error:", e)

## 2. Tool schemas (`name`, `description`, `input_schema`)

In [ ]:
tools = [
    {
        "name": "get_current_datetime",
        "description": "Returns the current local date and time. Use it whenever the user mentions a relative time such as 'in 2 hours' or 'tomorrow'. Returns a string formatted with date_format.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "strftime format of the result. Default '%Y-%m-%d %H:%M:%S'.",
                }
            },
        },
    },
    {
        "name": "add_duration_to_datetime",
        "description": "Adds a duration to a datetime and returns the new datetime as 'YYYY-MM-DD HH:MM:SS'. Use it to turn a relative time into an absolute one.",
        "input_schema": {
            "type": "object",
            "properties": {
                "datetime_str": {"type": "string", "description": "Start datetime, ISO 8601 (YYYY-MM-DD HH:MM:SS)."},
                "duration": {"type": "number", "description": "How much to add. May be fractional."},
                "unit": {"type": "string", "enum": ["minutes", "hours", "days"], "description": "Unit of duration."},
            },
            "required": ["datetime_str", "duration", "unit"],
        },
    },
    {
        "name": "set_reminder",
        "description": "Sets a reminder at an absolute time. Call it last, after the time is known. Returns a confirmation string.",
        "input_schema": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "What to remind the user about."},
                "timestamp": {"type": "string", "description": "When to fire, ISO 8601 (YYYY-MM-DD HH:MM:SS)."},
            },
            "required": ["content", "timestamp"],
        },
    },
]

TOOL_FUNCTIONS = {
    "get_current_datetime": get_current_datetime,
    "add_duration_to_datetime": add_duration_to_datetime,
    "set_reminder": set_reminder,
}

## 3. Helpers and tool-running loop

In [ ]:
def add_user_message(messages, message):
    # message can be a string, a list of blocks, or a Message
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, message):
    content = message.content if hasattr(message, "content") else message
    messages.append({"role": "assistant", "content": content})


def chat(messages, system=None, tools=None):
    params = {"model": MODEL, "max_tokens": 4096, "messages": messages}
    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools
    return client.messages.create(**params)  # return the full message, not only text


def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")


def run_tools(message):
    # one tool_result per tool_use block, each with the matching tool_use_id
    results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        try:
            output = TOOL_FUNCTIONS[block.name](**block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output), "is_error": False})
        except Exception as e:
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True})
    return results


def run_conversation(messages):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)  # keep ALL blocks, not just text
        print(text_from_message(response))
        if response.stop_reason != "tool_use":
            return messages
        add_user_message(messages, run_tools(response))

## 4. Try it (needs `ANTHROPIC_API_KEY` in `.env`)

In [ ]:
messages = []
add_user_message(messages, "Remind me about the team meeting in 2 hours 30 minutes.")
messages = run_conversation(messages)

## Check yourself

- The reply can contain a `text` block and one or more `tool_use` blocks. Why append the whole `response.content`?
- What breaks if a `tool_result` has the wrong `tool_use_id`?
- Make `add_duration_to_datetime` raise on purpose: does Claude retry with corrected input?